In [3]:
"""
Framework para entrenamiento de modelos de deep learning en clasificación de imágenes médicas.
Este código se enfoca en la clasificación de osteoartritis de rodilla usando diferentes arquitecturas
de CNNs y Transformers.

Autor: [Tu nombre]
Fecha: 2025
"""

# ============================================================================
# IMPORTS
# ============================================================================

# Librerías estándar
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Métricas y validación
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    mean_absolute_error
)
from sklearn.model_selection import train_test_split

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0, ResNet152
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import (
    BatchNormalization, Conv2D, Dense, Dropout, Flatten, 
    GlobalAveragePooling2D, Lambda, MaxPooling2D, Permute, 
    Rescaling, LayerNormalization
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

# Transformers de Hugging Face
from transformers import (
    TFDeiTModel, TFSwinModel, TFViTModel, BeitConfig, 
    DeiTConfig, SwinConfig, ViTConfig
)
local=False
print("Imports realizados con éxito!")
# ============================================================================
# CONFIGURACIÓN GLOBAL
# ============================================================================

# Configuración de TensorFlow para reducir logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# Rutas de datos (configuradas para Kaggle)
TRAIN_DIR = '/kaggle/input/knee-osteoarthritis-dataset-with-severity/train'
VAL_DIR = '/kaggle/input/knee-osteoarthritis-dataset-with-severity/val'
TEST_DIR = '/kaggle/input/knee-osteoarthritis-dataset-with-severity/test'
CAT_DIR = '/kaggle/input/cat-knee/clean'

# Parámetros por defecto
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
SEED = 66

print("- Configuración inicial completada\n\n")
# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================
print("- Funciones auxiliares...")
def create_dataframe_from_directory(base_dir, regression=False):
    """
    Crea un DataFrame con las rutas de imágenes y sus etiquetas.
    
    Args:
        base_dir: Directorio base con subdirectorios por clase
        regression: Si True, las etiquetas son float, sino string
    
    Returns:
        DataFrame con columnas 'filename' y 'label'
    """
    data = []
    for label_str in sorted(os.listdir(base_dir)):
        label_path = os.path.join(base_dir, label_str)
        if not os.path.isdir(label_path):
            continue
            
        # Convierte etiqueta según el tipo de problema
        label = float(label_str) if regression else label_str
        
        # Añade todas las imágenes de esta clase
        for fname in os.listdir(label_path):
            data.append({
                'filename': os.path.join(label_str, fname),
                'label': label
            })
    
    return pd.DataFrame(data)


def get_num_classes(tarea, regression=False):
    """
    Determina el número de clases según la tarea.
    
    Args:
        tarea: Número de tarea (0-5)
        regression: Si es regresión
    
    Returns:
        Número de clases
    """
    if regression:
        return 1
    elif tarea == 1:
        return 3  # Clases 0-2 vs 3 vs 4
    elif tarea > 1:
        return 2  # Binaria: clase 0 vs clase objetivo
    else:
        return 5  # Multiclase: todas las clases KL (0-4)


def apply_task_transform(df, tarea):
    """
    Aplica transformaciones de etiquetas según la tarea.
    
    Args:
        df: DataFrame con columna 'label'
        tarea: Número de tarea (0-5)
    
    Returns:
        DataFrame con etiquetas transformadas
    """
    df_proc = df.copy()
    df_proc['label'] = df_proc['label'].astype(int)
    
    if tarea == 0:
        # Multiclase: mantener todas las clases KL (0-4)
        pass
    elif tarea == 1:
        # Tres clases: [0,1,2] vs [3] vs [4]
        df_proc['label'] = df_proc['label'].apply(
            lambda x: 0 if x in [0,1,2] else 1 if x == 3 else 2
        )
    elif tarea in [2,3,4,5]:
        # Binaria: clase 0 vs clase objetivo
        target = tarea - 1
        df_proc = df_proc[df_proc['label'].isin([0, target])]
        df_proc['label'] = df_proc['label'].apply(
            lambda x: 1 if x == target else 0
        )
    else:
        raise ValueError(f"Tarea {tarea} no soportada. Usar 0-5.")
    
    return df_proc.reset_index(drop=True)


def balance_dataset(df, seed=SEED):
    """
    Balancea el dataset mediante oversampling.
    
    Args:
        df: DataFrame con columna 'label'
        seed: Semilla aleatoria
    
    Returns:
        DataFrame balanceado
    """
    counts = df['label'].value_counts()
    max_count = counts.max()
    
    frames = []
    for cls, cnt in counts.items():
        df_cls = df[df['label'] == cls]
        if cnt < max_count:
            # Sobremuestreo con reemplazo
            df_cls = df_cls.sample(max_count, replace=True, random_state=seed)
        frames.append(df_cls)
    
    return pd.concat(frames).reset_index(drop=True)


# ============================================================================
# CALLBACKS Y EVALUACIÓN
# ============================================================================
print("- Callbacks y evaluación...")
class CatValidationCallback(tf.keras.callbacks.Callback):
    """
    Callback personalizado para validación en dataset de gatos.
    Evalúa el modelo en cada época y guarda el mejor.
    """
    
    def __init__(self, cat_generator, regression=False, filepath='best_model_cat.keras'):
        super().__init__()
        self.cat_generator = cat_generator
        self.regression = regression
        self.filepath = filepath
        self.epoch_metrics = []
        self.best_score = 0.0
        self.best_loss = np.inf

    def on_epoch_end(self, epoch, logs=None):
        """Evalúa el modelo al final de cada época."""
        preds = self.model.predict(self.cat_generator, verbose=0)

        if self.regression:
            # Para regresión: calcular MAE y accuracy sobre clases redondeadas
            y_pred = preds.flatten()
            y_true = self.cat_generator.labels
            loss = mean_absolute_error(y_true, y_pred)
            
            # Convertir a clases para calcular accuracy
            y_pred_class = np.clip(np.round(y_pred), 0, 4).astype(int)
            y_true_class = np.clip(np.round(y_true), 0, 4).astype(int)
            acc = accuracy_score(y_true_class, y_pred_class)
            score = acc
        else:
            # Para clasificación: evaluar loss y accuracy
            loss, acc = self.model.evaluate(self.cat_generator, verbose=0)
            score = acc

        # Guardar métricas
        self.epoch_metrics.append({
            'epoch': epoch + 1,
            'val_cat_loss': loss,
            'val_cat_accuracy': acc
        })

        # Mostrar resultados
        metric_name = "MAE" if self.regression else "Loss"
        print(f"🐾 [Cat Val] {metric_name}: {loss:.4f} | Accuracy: {acc:.4f}")

        # Guardar si hay mejora
        if score > self.best_score or (score == self.best_score and loss < self.best_loss):
            self.best_score = score
            self.best_loss = loss
            self.model.save(self.filepath)
            print(f"📦 Modelo mejorado guardado en {self.filepath}")


def get_callbacks(regression=False, cat_gen=None, finetuning=False):
    """
    Configura los callbacks para el entrenamiento.
    
    Args:
        regression: Si es un problema de regresión
        cat_gen: Generador para validación con gatos (opcional)
        finetuning: Si se está haciendo fine-tuning
    
    Returns:
        Lista de callbacks
    """
    # Configurar paciencia según el tipo de entrenamiento
    early_stopping_patience = 5 if finetuning else 10
    lr_patience = 3
    
    # Configurar checkpoint según el tipo de problema
    if regression:
        checkpoint = ModelCheckpoint(
            filepath='best_model.keras',
            monitor='val_loss',
            mode='min',
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        )
    else:
        checkpoint = ModelCheckpoint(
            filepath='best_model.keras',
            monitor='val_accuracy',
            mode='max',
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        )
    
    # Callbacks básicos
    callbacks = [
        EarlyStopping(patience=early_stopping_patience, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.5, patience=lr_patience, verbose=1),
        checkpoint
    ]
    
    # Añadir callback de validación con gatos si se proporciona
    if cat_gen:
        callbacks.append(CatValidationCallback(cat_gen, regression=regression))
    
    return callbacks


def evaluate_classification(y_true, y_pred, label_map=None, digits=4):
    """
    Evalúa un modelo de clasificación mostrando métricas y matriz de confusión.
    
    Args:
        y_true: Etiquetas verdaderas
        y_pred: Predicciones
        label_map: Nombres de las clases
        digits: Decimales para mostrar
    """
    # Generar nombres de clases si no se proporcionan
    if label_map is None:
        classes = np.unique(np.concatenate((y_true, y_pred)))
        label_map = [f'KL {i}' for i in classes]
    
    # Reporte de clasificación
    print(classification_report(y_true, y_pred, target_names=label_map, digits=digits))
    
    # Matriz de confusión
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_map, yticklabels=label_map)
    plt.xlabel('Predicción')
    plt.ylabel('Etiqueta verdadera')
    plt.title('Matriz de Confusión')
    plt.show()


def evaluate_model(model, regression, generator, label_map=None, digits=4):
    """
    Evalúa un modelo entrenado.
    
    Args:
        model: Modelo a evaluar
        regression: Si es regresión o clasificación
        generator: Generador de datos
        label_map: Nombres de las clases
        digits: Decimales para mostrar
    """
    preds = model.predict(generator)
    
    if regression:
        # Evaluación para regresión
        y_true_cont = np.array(generator.labels).flatten()
        y_pred_cont = np.array(preds).flatten()
        
        # Calcular MAE
        mae = mean_absolute_error(y_true_cont, y_pred_cont)
        print(f"Mean Absolute Error (MAE): {mae:.{digits}f}\n")
        
        # Gráfico de dispersión
        plt.figure(figsize=(6, 6))
        plt.scatter(y_pred_cont, y_true_cont, alpha=0.5)
        
        # Línea de identidad
        min_val = min(y_true_cont.min(), y_pred_cont.min())
        max_val = max(y_true_cont.max(), y_pred_cont.max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1)
        
        plt.xlabel("Predicción")
        plt.ylabel("Valor verdadero")
        plt.title("Predicción vs Valor verdadero")
        plt.grid(True)
        plt.show()
        
        # Evaluar también como clasificación (redondeando)
        y_true_class = np.clip(np.round(y_true_cont).astype(int), 0, None)
        y_pred_class = np.clip(np.round(y_pred_cont).astype(int), 0, None)
        
        if label_map is None:
            max_label = max(y_true_class.max(), y_pred_class.max())
            label_map = [f'KL {i}' for i in range(max_label + 1)]
        
        evaluate_classification(y_true_class, y_pred_class, label_map, digits)
    else:
        # Evaluación para clasificación
        y_pred = np.argmax(preds, axis=1)
        y_true = generator.classes
        
        if label_map is None:
            if hasattr(generator, 'class_indices'):
                sorted_items = sorted(generator.class_indices.items(), key=lambda x: x[1])
                label_map = [name for name, _ in sorted_items]
            else:
                classes = np.unique(np.concatenate((y_true, y_pred)))
                label_map = [f'KL {i}' for i in classes]
        
        evaluate_classification(y_true, y_pred, label_map, digits)


# ============================================================================
# GENERACIÓN DE DATOS
# ============================================================================
print("- Generación de datos...")
def get_datagen(tarea, path, regression=False, img_size=(224, 224), 
                batch_size=32, seed=42, augment=False, balanced=False):
    """
    Crea un generador de datos para una tarea específica.
    
    Args:
        tarea: Número de tarea (0-5)
        path: Ruta al directorio de datos
        regression: Si es regresión o clasificación
        img_size: Tamaño de las imágenes
        batch_size: Tamaño del lote
        seed: Semilla aleatoria
        augment: Si aplicar data augmentation
        balanced: Si balancear las clases
    
    Returns:
        Generador de datos de Keras
    """
    print(f"[get_datagen] tarea={tarea}, regression={regression}, augment={augment}, balanced={balanced}")

    # Configurar data augmentation
    if augment:
        datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=10,
            width_shift_range=0.1,
            height_shift_range=0.1,
            zoom_range=0.1,
            horizontal_flip=True
        )
        shuffle = True
    else:
        datagen = ImageDataGenerator(rescale=1./255)
        shuffle = False

    # Crear DataFrame con las imágenes
    df = create_dataframe_from_directory(path, regression=regression)
    print(f"[get_datagen] Imágenes encontradas: {len(df)}")

    # Aplicar transformaciones según la tarea
    df_proc = apply_task_transform(df, tarea)
    print(f"[get_datagen] Tras tarea: {df_proc['label'].value_counts().to_dict()}")

    # Balancear dataset si es necesario
    if balanced:
        df_proc = balance_dataset(df_proc, seed)
        print(f"[get_datagen] Tras balance: {df_proc['label'].value_counts().to_dict()}")

    # Configurar el generador según el tipo de problema
    if regression:
        df_proc['label'] = df_proc['label'].astype(float)
        class_mode = 'raw'
    else:
        df_proc['label'] = df_proc['label'].astype(str)
        class_mode = 'categorical'

    # Crear generador
    gen = datagen.flow_from_dataframe(
        dataframe=df_proc,
        directory=path,
        x_col='filename',
        y_col='label',
        target_size=img_size,
        batch_size=batch_size,
        class_mode=class_mode,
        shuffle=shuffle,
        seed=seed
    )

    print(f"[get_datagen] Generador creado: mode={class_mode}, muestras={gen.samples}")
    return gen


def get_data_split_generators(tarea, path, regression=False, test_size=0, val_size=0.3,
                             random_state=42, augment=False, balanced=False,
                             img_size=(224, 224), batch_size=32, seed=42):
    """
    Divide los datos en train/val/test y crea generadores para cada uno.
    
    Args:
        tarea: Número de tarea (0-5)
        path: Ruta al directorio de datos
        regression: Si es regresión
        test_size: Proporción para test
        val_size: Proporción para validación
        random_state: Semilla para división
        augment: Si aplicar augmentation (solo en train)
        balanced: Si balancear train
        img_size: Tamaño de imágenes
        batch_size: Tamaño del lote
        seed: Semilla para generadores
    
    Returns:
        train_gen, val_gen, test_gen
    """
    # Cargar datos base
    df = create_dataframe_from_directory(path, regression=False)
    df['label'] = df['label'].astype(int)

    # Dividir en train+val vs test
    if test_size > 0.0:
        df_trainval, df_test = train_test_split(
            df, test_size=test_size, random_state=random_state,
            stratify=df['label']
        )
    else:
        df_trainval = df
        df_test = None

    # Dividir train vs val
    val_rel = val_size / (1 - test_size)
    df_train, df_val = train_test_split(
        df_trainval, test_size=val_rel, random_state=random_state,
        stratify=df_trainval['label']
    )

    # Aplicar transformaciones de tarea
    df_train = apply_task_transform(df_train, tarea)
    df_val = apply_task_transform(df_val, tarea)
    if df_test is not None:
        df_test = apply_task_transform(df_test, tarea)

    # Balancear train si es necesario
    if balanced:
        df_train = balance_dataset(df_train, random_state)
        print(f"[DEBUG] Dataset balanceado: {len(df_train)} muestras de entrenamiento")

    # Configurar generadores
    if regression:
        for df_subset in [df_train, df_val, df_test]:
            if df_subset is not None:
                df_subset['label'] = df_subset['label'].astype(float)
        class_mode = 'raw'
    else:
        for df_subset in [df_train, df_val, df_test]:
            if df_subset is not None:
                df_subset['label'] = df_subset['label'].astype(str)
        class_mode = 'categorical'

    # Crear generadores de imágenes
    if augment:
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=10,
            width_shift_range=0.1,
            height_shift_range=0.1,
            zoom_range=0.1,
            horizontal_flip=True
        )
        train_shuffle = True
    else:
        train_datagen = ImageDataGenerator(rescale=1./255)
        train_shuffle = False

    eval_datagen = ImageDataGenerator(rescale=1./255)

    # Crear generadores
    train_gen = train_datagen.flow_from_dataframe(
        dataframe=df_train, directory=path, x_col='filename', y_col='label',
        target_size=img_size, batch_size=batch_size, class_mode=class_mode,
        shuffle=train_shuffle, seed=seed
    )

    val_gen = eval_datagen.flow_from_dataframe(
        dataframe=df_val, directory=path, x_col='filename', y_col='label',
        target_size=img_size, batch_size=batch_size, class_mode=class_mode,
        shuffle=False, seed=seed
    )

    test_gen = None
    if df_test is not None:
        test_gen = eval_datagen.flow_from_dataframe(
            dataframe=df_test, directory=path, x_col='filename', y_col='label',
            target_size=img_size, batch_size=batch_size, class_mode=class_mode,
            shuffle=False, seed=seed
        )

    return train_gen, val_gen, test_gen

# ============================================================================
# ARQUITECTURAS DE MODELOS
# ============================================================================
print("- Arquitectura de modelos")
def cnn_simple(num_classes=5, regression=False):
    """
    CNN simple con una capa convolucional.
    
    Args:
        num_classes: Número de clases de salida
        regression: Si es regresión o clasificación
    
    Returns:
        Modelo compilado
    """
    activation = 'linear' if regression else 'softmax'
    
    model = Sequential([
        Conv2D(16, (3, 3), activation='relu', input_shape=(224, 224, 3)),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(num_classes, activation=activation)
    ], name="CNN_Simple")
    
    return model


def cnn_medium(num_classes=5, regression=False):
    """
    CNN de tamaño mediano con dos capas convolucionales.
    
    Args:
        num_classes: Número de clases de salida
        regression: Si es regresión o clasificación
    
    Returns:
        Modelo compilado
    """
    activation = 'linear' if regression else 'softmax'
    
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(224, 224, 3)),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(num_classes, activation=activation)
    ], name="CNN_Medium")
    
    return model


def cnn_deep(num_classes=5, regression=False):
    """
    CNN profunda con múltiples bloques convolucionales y regularización.
    
    Args:
        num_classes: Número de clases de salida
        regression: Si es regresión o clasificación
    
    Returns:
        Modelo compilado
    """
    activation = 'linear' if regression else 'softmax'
    
    model = Sequential(name="CNN_Deep")
    
    # Bloque 1
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(224, 224, 3)))
    model.add(BatchNormalization())
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.25))
    
    # Bloque 2
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.25))
    
    # Bloque 3
    model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.40))
    
    # Clasificador
    model.add(Flatten())
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.50))
    model.add(Dense(num_classes, activation=activation))
    
    return model


def efficientnet_model(num_classes=5, regression=False, frozen=False):
    """
    Modelo basado en EfficientNetB0 preentrenado.
    
    Args:
        num_classes: Número de clases de salida
        regression: Si es regresión o clasificación
        frozen: Si congelar pesos preentrenados
    
    Returns:
        Modelo compilado
    """
    activation = 'linear' if regression else 'softmax'
    
    # Base preentrenada
    base = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3)
    )
    
    if frozen:
        base.trainable = False
    
    # Modelo completo
    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dropout(0.50),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.50),
        Dense(num_classes, activation=activation)
    ], name="EfficientNetB0_Custom")
    
    return model


def resnet_model(num_classes=5, regression=False, frozen=False):
    """
    Modelo basado en ResNet152 preentrenado.
    
    Args:
        num_classes: Número de clases de salida
        regression: Si es regresión o clasificación
        frozen: Si congelar pesos preentrenados
    
    Returns:
        Modelo compilado
    """
    activation = 'linear' if regression else 'softmax'
    
    # Base preentrenada
    base = ResNet152(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3)
    )
    
    if frozen:
        base.trainable = False
    
    # Modelo completo
    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dropout(0.5),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes, activation=activation)
    ], name="ResNet152_Custom")
    
    return model



def transformer_simple(num_classes=5, regression=False, frozen=False, pretrained=True):
    """
    Transformer simple mejorado basado en DeiT.
    Optimizado para imágenes médicas con cabeza simplificada y técnicas específicas.
    
    Args:
        num_classes: Número de clases de salida
        regression: Si es regresión o clasificación
        frozen: Si congelar backbone
        pretrained: Si usar pesos preentrenados
    
    Returns:
        Modelo compilado
    """
    activation = 'linear' if regression else 'softmax'
    
    # Configuración del backbone DeiT con ajustes
    config = DeiTConfig.from_pretrained(
        'facebook/deit-base-patch16-224',
        num_labels=num_classes,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1
    )
    
    backbone = (
        TFDeiTModel.from_pretrained('facebook/deit-base-patch16-224', config=config)
        if pretrained else TFDeiTModel(config)
    )
    
    backbone.trainable = not frozen
    
    # Función para extraer características del transformer
    def extract_features(pixel_values):
        """Extrae características del transformer de manera más robusta."""
        # Normalizar entrada a rango esperado por DeiT
        pixel_values = tf.cast(pixel_values, tf.float32)
        
        # DeiT espera valores en [-1, 1] aproximadamente
        pixel_values = (pixel_values - 127.5) / 127.5
        
        # Obtener salida del transformer
        outputs = backbone(pixel_values=pixel_values, training=tf.keras.backend.learning_phase())
        
        # Usar CLS token (primer token) como representación
        cls_token = outputs.last_hidden_state[:, 0, :]
        
        return cls_token
    
    # Cabeza simplificada y más robusta
    model = Sequential([
        # Preprocesamiento optimizado
        Lambda(lambda x: tf.cast(x, tf.float32), input_shape=(224, 224, 3)),
        
        # Extraer características
        Lambda(extract_features),
        
        # Cabeza de clasificación simplificada
        LayerNormalization(),
        Dropout(0.3),
        Dense(128, activation='gelu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        LayerNormalization(),
        Dropout(0.2),
        Dense(num_classes, activation=activation)
    ], name="Transformer_Simple_v2")
    
    return model


def transformer_advanced(num_classes=5, regression=False, frozen=False, pretrained=True):
    """
    Transformer avanzado mejorado con técnicas específicas para imágenes médicas.
    Incluye regularización adaptativa, cabeza multi-escala y técnicas de estabilización.
    
    Args:
        num_classes: Número de clases de salida
        regression: Si es regresión o clasificación
        frozen: Si congelar backbone
        pretrained: Si usar pesos preentrenados
    
    Returns:
        Modelo compilado
    """
    activation = 'linear' if regression else 'softmax'
    
    # Configuración del backbone DeiT con ajustes avanzados
    config = DeiTConfig.from_pretrained(
        'facebook/deit-base-patch16-224',
        num_labels=num_classes,
        hidden_dropout_prob=0.15,
        attention_probs_dropout_prob=0.1,
        layer_norm_eps=1e-6
    )
    
    backbone = (
        TFDeiTModel.from_pretrained('facebook/deit-base-patch16-224', config=config)
        if pretrained else TFDeiTModel(config)
    )
    
    backbone.trainable = not frozen
    
    # Función para extraer características multi-escala
    def extract_multi_scale_features(pixel_values):
        """Extrae características multi-escala del transformer."""
        # Normalizar entrada correctamente
        pixel_values = tf.cast(pixel_values, tf.float32)
        pixel_values = (pixel_values - 127.5) / 127.5
        
        # Obtener salida del transformer
        outputs = backbone(pixel_values=pixel_values, training=tf.keras.backend.learning_phase())
        
        # Extraer CLS token y tokens de patch representativos
        cls_token = outputs.last_hidden_state[:, 0, :]  # CLS token
        patch_tokens = outputs.last_hidden_state[:, 1:, :]  # Tokens de patch
        
        # Pooling global de los tokens de patch
        global_pool = tf.reduce_mean(patch_tokens, axis=1)
        
        # Combinar CLS token con pooling global
        combined_features = tf.concat([cls_token, global_pool], axis=-1)
        
        return combined_features
    
    # Cabeza avanzada con arquitectura multi-escala
    model = Sequential([
        # Preprocesamiento optimizado
        Lambda(lambda x: tf.cast(x, tf.float32), input_shape=(224, 224, 3)),
        
        # Extraer características multi-escala
        Lambda(extract_multi_scale_features),
        
        # Cabeza de clasificación avanzada
        LayerNormalization(),
        Dropout(0.3),
        
        # Primera rama - procesamiento profundo
        Dense(512, activation='gelu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        LayerNormalization(),
        Dropout(0.25),
        
        # Segunda rama - refinamiento
        Dense(256, activation='gelu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        LayerNormalization(),
        Dropout(0.2),
        
        # Rama final - predicción
        Dense(64, activation='gelu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        LayerNormalization(),
        Dropout(0.15),
        
        # Salida final
        Dense(num_classes, activation=activation)
    ], name="Transformer_Advanced_v2")
    
    return model


def transformer_hybrid(num_classes=5, regression=False, frozen=False, pretrained=True):
    """
    Modelo híbrido CNN+Transformer optimizado para imágenes médicas.
    Combina extracción de características CNN con atención transformer.
    
    Args:
        num_classes: Número de clases de salida
        regression: Si es regresión o clasificación
        frozen: Si congelar backbone
        pretrained: Si usar pesos preentrenados
    
    Returns:
        Modelo compilado
    """
    activation = 'linear' if regression else 'softmax'
    
    # Configuración del backbone DeiT
    config = DeiTConfig.from_pretrained(
        'facebook/deit-base-patch16-224',
        num_labels=num_classes,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1
    )
    
    transformer_backbone = (
        TFDeiTModel.from_pretrained('facebook/deit-base-patch16-224', config=config)
        if pretrained else TFDeiTModel(config)
    )
    
    transformer_backbone.trainable = not frozen
    
    # Función para extraer características híbridas
    def extract_hybrid_features(pixel_values):
        """Extrae características combinando CNN y transformer."""
        pixel_values = tf.cast(pixel_values, tf.float32)
        
        # Branch 1: Características CNN locales
        cnn_features = Conv2D(128, (3, 3), activation='relu', padding='same')(pixel_values)
        cnn_features = BatchNormalization()(cnn_features)
        cnn_features = MaxPooling2D((2, 2))(cnn_features)
        cnn_features = Conv2D(256, (3, 3), activation='relu', padding='same')(cnn_features)
        cnn_features = BatchNormalization()(cnn_features)
        cnn_features = GlobalAveragePooling2D()(cnn_features)
        
        # Branch 2: Características Transformer globales
        transformer_input = (pixel_values - 127.5) / 127.5
        transformer_outputs = transformer_backbone(pixel_values=transformer_input, training=tf.keras.backend.learning_phase())
        transformer_features = transformer_outputs.last_hidden_state[:, 0, :]  # CLS token
        
        # Combinar características
        combined_features = tf.concat([cnn_features, transformer_features], axis=-1)
        
        return combined_features
    
    # Modelo híbrido
    model = Sequential([
        # Preprocesamiento
        Lambda(lambda x: tf.cast(x, tf.float32), input_shape=(224, 224, 3)),
        
        # Extraer características híbridas
        Lambda(extract_hybrid_features),
        
        # Cabeza de fusión
        LayerNormalization(),
        Dropout(0.3),
        
        # Procesamiento de características fusionadas
        Dense(512, activation='gelu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        LayerNormalization(),
        Dropout(0.2),
        
        Dense(128, activation='gelu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        LayerNormalization(),
        Dropout(0.1),
        
        # Salida final
        Dense(num_classes, activation=activation)
    ], name="Transformer_Hybrid")
    
    return model


# ============================================================================
# FUNCIONES DE ENTRENAMIENTO
# ============================================================================
print("- Entrenamiento...")
def train_model(model, tarea=0, regression=False, frozen=False, finetuning=False, 
                balanced=False, batch_size=32, epochs=50, verbose=1, 
                train_dir=TRAIN_DIR, val_dir=VAL_DIR, run_eagerly=False):
    """
    Función principal para entrenar modelos.
    
    Args:
        model: Modelo a entrenar
        tarea: Número de tarea (0-5)
        regression: Si es regresión
        frozen: Si congelar capas preentrenadas
        finetuning: Si es fine-tuning
        balanced: Si balancear dataset
        batch_size: Tamaño del lote
        epochs: Número de épocas
        verbose: Nivel de verbosidad
        train_dir: Directorio de entrenamiento
        val_dir: Directorio de validación
        run_eagerly: Si ejecutar en modo eager
    
    Returns:
        model: Modelo entrenado
        history: Historial de entrenamiento
    """
    # Configurar parámetros
    num_classes = get_num_classes(tarea, regression)
    loss = 'mean_squared_error' if regression else 'categorical_crossentropy'
    metrics = ['mae'] if regression else ['accuracy']
    
    # Configurar optimizador
    lr = 0.0001 if finetuning else 0.001
    optimizer = Adam(learning_rate=lr)
    
    # Crear generadores de datos
    train_gen = get_datagen(
        tarea=tarea, path=train_dir, regression=regression,
        batch_size=batch_size, augment=True, balanced=balanced
    )
    
    val_gen = get_datagen(
        tarea=tarea, path=val_dir, regression=regression,
        batch_size=batch_size, augment=False, balanced=False
    )
    
    # Configurar ejecución
    if run_eagerly:
        tf.config.run_functions_eagerly(True)
    
    # Compilar modelo
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=metrics,
        run_eagerly=run_eagerly
    )
    
    # Configurar callbacks
    callbacks = get_callbacks(regression=regression, finetuning=finetuning)
    
    # Mostrar información del entrenamiento
    print_training_info(tarea, regression, frozen, finetuning, balanced, batch_size)
    
    # Entrenar modelo
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        verbose=verbose,
        callbacks=callbacks
    )
    
    # Evaluar modelo
    evaluate_model(model, regression=regression, generator=val_gen)
    
    # Cargar mejores pesos y re-evaluar
    model.load_weights('best_model.keras')
    evaluate_model(model, regression=regression, generator=val_gen)
    
    return model, history


    
def train_with_cat_validation(model, tarea=0, regression=False, frozen=False, 
                             finetuning=False, balanced=False, batch_size=32, 
                             epochs=50, verbose=1, seed=43, cat_dir=CAT_DIR):
    """
    Función de entrenamiento con validación adicional en dataset de gatos.
    
    Args:
        model: Modelo a entrenar
        tarea: Número de tarea (0-5)
        regression: Si es regresión
        frozen: Si congelar capas preentrenadas
        finetuning: Si es fine-tuning
        balanced: Si balancear dataset
        batch_size: Tamaño del lote
        epochs: Número de épocas
        verbose: Nivel de verbosidad
        seed: Semilla aleatoria
        cat_dir: Directorio del dataset de gatos
    
    Returns:
        model: Modelo entrenado
        history: Historial de entrenamiento
    """
    # Configurar parámetros
    num_classes = get_num_classes(tarea, regression)
    loss = 'mean_squared_error' if regression else 'categorical_crossentropy'
    metrics = ['mae'] if regression else ['accuracy']
    
    # Configurar optimizador
    lr = 0.00001 if finetuning else 0.001
    optimizer = Adam(learning_rate=lr)
    
    # Crear generadores de datos
    train_gen, val_gen, test_gen = get_data_split_generators(
        tarea=tarea, path=cat_dir, regression=regression,
        batch_size=batch_size, augment=True, balanced=balanced,
        seed=seed
    )
    
    # Compilar modelo
    model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
    
    # Configurar callbacks (incluyendo validación con gatos)
    callbacks = get_callbacks(regression=regression, cat_gen=val_gen)
    
    # Mostrar información del entrenamiento
    print_training_info(tarea, regression, frozen, finetuning, balanced, batch_size)
    
    # Entrenar modelo
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        verbose=verbose,
        callbacks=callbacks
    )
    
    # Evaluar modelo
    model.load_weights('best_model.keras')
    evaluate_model(model, regression=regression, generator=val_gen)
    
    return model, history


def print_training_info(tarea, regression, frozen, finetuning, balanced, batch_size):
    """
    Muestra información del entrenamiento.
    
    Args:
        tarea: Número de tarea
        regression: Si es regresión
        frozen: Si está congelado
        finetuning: Si es fine-tuning
        balanced: Si está balanceado
        batch_size: Tamaño del lote
    """
    separator = "=" * 60
    print(f"\n{separator}")
    print(f"🚀 Iniciando entrenamiento para la tarea: {tarea}")
    print(f"{'-' * 60}")
    print(f"   • Tipo de tarea     : {'Regresión' if regression else 'Clasificación'}")
    print(f"   • Pesos congelados  : {'Sí' if frozen else 'No'}")
    print(f"   • Fine-tuning activo: {'Sí' if finetuning else 'No'}")
    print(f"   • Dataset balanceado: {'Sí' if balanced else 'No'}")
    print(f"   • Tamaño batch      : {batch_size}")
    print(f"{separator}\n")

Imports realizados con éxito!
- Configuración inicial completada


- Funciones auxiliares...
- Callbacks y evaluación...
- Generación de datos...
- Arquitectura de modelos
- Entrenamiento...


In [6]:
local=True

In [ ]:
# Modelo EfficientNetB0
tarea = 0
regression = False  # Cambiar a True si es regresión

num_classes = 5 if tarea == 0 else 2
epochs = 10 if local else 100

# Entrenar modelo con datos de gatos
print("🐱 Iniciando entrenamiento con dataset de gatos...")

model = efficientnet_model(num_classes=num_classes, regression=regression)

# Splitear los datos de CAT_DIR en train/val
train_gen, val_gen, test_gen = get_data_split_generators(
    tarea=tarea, path=CAT_DIR, regression=regression,
    batch_size=BATCH_SIZE, augment=True, balanced=True,
    seed=42  # Semilla para reproducibilidad
)
# Entrenar modelo usando los datos de gatos como entrenamiento
model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mean_squared_error' if regression else 'categorical_crossentropy',
        metrics=['mae'] if regression else ['accuracy']
    )
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=epochs,
    verbose=1 if local else 0,
    callbacks=[]
)

print("✅ Entrenamiento completado!")
print("📊 El modelo ha sido evaluado y guardado como 'best_model.keras'")
print("🔍 También se guardó el mejor modelo en validación de gatos como 'best_model_cat.keras'")
# Evaluar el modelo
evaluate_model(model, regression, CAT_DIR, label_map=None, digits=4)

🐱 Iniciando entrenamiento con dataset de gatos...
[DEBUG] Dataset balanceado: 105 muestras de entrenamiento
Found 105 validated image filenames belonging to 5 classes.
Found 25 validated image filenames belonging to 5 classes.
Epoch 1/10


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


In [ ]:
## Entrenar el modelo utilizando datos de gatos para validación

model = efficientnet_model(num_classes=num_classes, regression=regression)

trained_model, history = train_with_cat_validation(
    model=model,
    tarea=tarea,
    regression=regression,
    frozen=False,  # No congelar capas para permitir aprendizaje completo
    finetuning=False,  # Entrenamiento inicial (no fine-tuning)
    balanced=True,  # Balancear dataset para mejor entrenamiento
    batch_size=BATCH_SIZE,  # Usar batch size definido (8)
    epochs=50,
    verbose=1,
    seed=42,  # Semilla para reproducibilidad
    cat_dir=CAT_DIR  # Usar datos de gatos para validación
)

print("✅ Entrenamiento con validación de gatos completado!")
print("📊 El modelo ha sido evaluado y guardado como 'best_model.keras'")
print("🔍 También se guardó el mejor modelo en validación de gatos como 'best_model_cat.keras'")

# Evaluar el modelo
evaluate_model(model, regression, CAT_DIR, label_map=None, digits=4)

# ============================================================================

In [ ]:
# Cargar el mejor modelo de validación de gatos
model.load_weights('best_model_cat.keras')
print("📂 Mejor modelo de validación de gatos cargado.")

# Entrenar con los gatos otra vez

trained_model, history = train_with_cat_validation(
    model=model,
    tarea=tarea,
    regression=regression,
    frozen=False,  # No congelar capas para permitir aprendizaje completo
    finetuning=False,  # Entrenamiento inicial (no fine-tuning)
    balanced=True,  # Balancear dataset para mejor entrenamiento
    batch_size=BATCH_SIZE,  # Usar batch size definido (8)
    epochs=50,
    verbose=1,
    seed=42,  # Semilla para reproducibilidad
    cat_dir=CAT_DIR  # Usar datos de gatos para validación
)

print("✅ Entrenamiento con validación de gatos completado!")
print("📊 El modelo ha sido evaluado y guardado como 'best_model.keras'")
print("🔍 También se guardó el mejor modelo en validación de gatos como 'best_model_cat.keras'")

🐱 Iniciando entrenamiento con dataset de gatos...
[DEBUG] Dataset balanceado: 105 muestras de entrenamiento
Found 105 validated image filenames belonging to 5 classes.
Found 25 validated image filenames belonging to 5 classes.


ValueError: You must call `compile()` before using the model.